## FunctionGraph & Function LaText to parquet
This notebook shows the final version of the file used to generate the dataset: [Croc-Prog-HF/Simplified_FunctionGraph-LaTeX](https://huggingface.co/datasets/Croc-Prog-HF/Simplified_FunctionGraph-LaTeX)<br/>

In [ ]:
!pip install pandas pyarrow
!pip install datasets
!pip install antlr4-python3-runtime==4.11

In [ ]:
import random
from sympy import symbols, sqrt, log, sin, cos, tan, exp, init_printing, latex
import numpy as np
import io
import time
import matplotlib.pyplot as plt
from sympy import lambdify, symbols, latex
from sympy.parsing.latex import parse_latex
import pandas as pd
from datasets import Dataset, Features, Value, Image

# ==============
# Formula latext
# ==============
def GenFx():
  init_printing(use_latex=True)
  x = symbols('x')
  y = symbols('y')

  def genera_monomio():
      scelta_base = random.randint(1, 4)

      if scelta_base == 1: # Caso Radice
          op = random.choice([
              x,
              log(x),
              sin(x),
              cos(x),
              tan(x),
              sin(x)/cos(x),
              sin(x)/tan(x),
              cos(x)/sin(x),
              cos(x)/tan(x)
          ])
          return sqrt(op)

      elif scelta_base == 2: # Caso Logaritmo
          base = random.choice([2, 3, 4, 5, 6, 7, 8, 9, 10, 'e'])
          arg_op = random.choice([sin(x), cos(x), tan(x)])
          if base == 'e':
              return log(arg_op)
          else:
              return log(arg_op, base)

      elif scelta_base == 3: # Caso Goniometria
          op = random.choice([
              sin(x),
              cos(x),
              tan(x),
              1/sin(x),
              1/cos(x),
              1/tan(x),
              exp(x),
              sin(x)/cos(x),
          ])
          return op

      elif scelta_base == 4: # Caso Potenza
          n = random.randint(2, 15)
          base_op = random.choice([
              sqrt(x),
              log(x, random.randint(2, 10)),
              random.randint(2, 500)
          ])
          return base_op**n

  # 1. Sceglie il numero di monomi
  n_mnm = random.randint(2, 6)
  # 2. Genera la funzione usando operatori casuali (+, -, *, /)
  monomi = [genera_monomio() for _ in range(n_mnm)]
  f_custom = monomi[0]

  for i in range(1, n_mnm):
      op_scelto = random.choice(['+', '-', '*', '/'])
      if op_scelto == '+':
          f_custom = f_custom + monomi[i]
      elif op_scelto == '-':
          f_custom = f_custom - monomi[i]
      elif op_scelto == '*':
          f_custom = f_custom * monomi[i]
      elif op_scelto == '/':
          f_custom = f_custom / monomi[i]
  #print(f"Numero di monomi (n_mnm): {n_mnm}")
  #display(f_custom)
  print("\nLaTeX grezzo:")
  print(latex(f_custom))
  return f_custom

# =============
# Generatore di grafici
# =============
def GenGraph(f_input):
  # Variabile simbolica
  x_sym = symbols('x')

  try:
      # Check if input is a string (LaTeX) or already a sympy expression
      """if isinstance(f_input, str):
          f_expr = parse_latex(f_input)
      else:"""
      f_expr = f_input

      if f_expr is None:
          raise ValueError("NULL")

      # Creazione della funzione numerica
      # We use 'numpy' as the primary module for lambdify
      # f_num = lambdify(x_sym, f_expr, modules=['numpy', 'sympy'])
      f_num = lambdify(x_sym, f_expr, modules=['numpy'])
      # f_vec = np.vectorize(f_num)

      # Definizione del dominio per il grafico
      x_vals = np.linspace(0.1, 10, 1000)
      # Calcolo dei valori y
      y_vals = f_num(x_vals)

      # Converti in array numpy "vero"
      y_vals = np.array(y_vals, dtype=np.complex128)
      y_vals = np.real(y_vals) # Prendi solo parte reale
      y_vals[~np.isfinite(y_vals)] = np.nan # Pulisci valori problematici

      if np.all(np.isnan(y_vals)):
          print("NULL")
          return None
      else:
          plt.figure(figsize=(10, 6))
          plt.plot(x_vals, y_vals, label=f"$f(x) = {latex(f_expr)}$")
          plt.axhline(0, color='black', linewidth=0.5)
          plt.axvline(0, color='black', linewidth=0.5)
          plt.grid(True, linestyle='--', alpha=0.7)

          # salvataggio
          buf = io.BytesIO()
          plt.savefig(buf, format='png')
          buf.seek(0)

          plt.show()
          plt.close()
      return buf.getvalue()
  except Exception as e:
      print(f"Error in GenGraph: {e}")
      return None


TIMER = 10  # Secondi di esecuzione
start_time = time.time()
data_list = []

while time.time() - start_time < TIMER:
    f_expr = GenFx()
    latex_str = latex(f_expr)
    img_bytes = GenGraph(f_expr)

    if img_bytes is not None:
        data_list.append({
            "graph": {
                "bytes": img_bytes,
                "path": None
            },
            "latex_formula": latex_str
        })

df = pd.DataFrame(data_list)
features = Features({
    "graph": Image(),
    "latex_formula": Value("string")
})
dataset = Dataset.from_pandas(df, features=features)
dataset.to_parquet("funzioni.parquet")
print(f"Salvate {len(dataset)} funzioni in funzioni.parquet con metadata HF")